# **深入浅出理解生成对抗网络 (GAN)**

## **GAN 概述**

生成对抗网络（Generative Adversarial Networks，简称 GAN）是一种深度学习模型。它通过两个神经网络 —— 生成器（Generator）和判别器（Discriminator）之间的对抗训练来生成新的、与训练数据类似的数据。

## **GAN 工作原理**

GAN 的核心思想是让生成器和判别器进行博弈：

- 生成器（G）：接收随机噪声作为输入，生成模拟的样本。生成器的目标是生成足够逼真的数据，以 “欺骗” 判别器。
- 判别器（D）：接收输入数据（可能是真实样本或生成样本），输出该样本为真实样本的概率。判别器的目标是准确区分真实样本和生成样本。

<center><img src="images/gan-arch-demo.jpg" style="width:80%"></center>

## **GAN 训练对抗过程**

- 生成器不断改进生成的数据，试图让判别器将其误判为真实数据。
- 判别器则不断优化，提高对真实数据和生成数据的区分能力。这种对抗过程可以类比为一个 “博弈游戏”，最终达到纳什均衡，生成器能够生成高质量的假数据，就如同图像生成模型中生成的假人脸。

## GAN 示例：生成手写数字图像

使用 MNIST 数据集，这是一个包含手写数字图像的经典数据集。生成器将生成看起来像手写数字的图像，判别器将尝试区分真实图像和生成图像。

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torchvision.utils import make_grid
import matplotlib.pyplot as plt

In [3]:
# 设置设备
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [4]:
# 超参数
batch_size = 64
learning_rate = 0.0002
num_epochs = 5
nz = 100  # 噪声向量的维度

In [5]:
# 数据加载与预处理
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])
train_dataset = datasets.MNIST(root='./data', train=True, transform=transform, download=True)
train_loader = torch.utils.data.DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)

In [6]:
# 定义生成器
class Generator(nn.Module):
    def __init__(self):
        super(Generator, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(nz, 256),
            nn.ReLU(True),
            nn.Linear(256, 512),
            nn.ReLU(True),
            nn.Linear(512, 1024),
            nn.ReLU(True),
            nn.Linear(1024, 28 * 28),
            nn.Tanh()  # 输出范围为 [-1, 1]
        )
    def forward(self, x):
        return self.model(x).view(-1, 1, 28, 28)

In [ ]:
# 定义判别器
class Discriminator(nn.Module):
    def __init__(self):
        super(Discriminator, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(28 * 28, 512),
            nn.LeakyReLU(0.2),
            nn.Linear(512, 256),
            nn.LeakyReLU(0.2),
            nn.Linear(256, 1),
            nn.Sigmoid()  # 输出范围为 [0, 1]
        )
    def forward(self, x):
        x = x.view(-1, 28 * 28)
        return self.model(x)

In [7]:
# 初始化模型
generator = Generator().to(device)
discriminator = Discriminator().to(device)

In [8]:
# 定义损失函数和优化器
criterion = nn.BCELoss()
g_optimizer = optim.Adam(generator.parameters(), lr=learning_rate)
d_optimizer = optim.Adam(discriminator.parameters(), lr=learning_rate)

In [9]:
# 训练过程
for epoch in range(num_epochs):
    for i, (images, _) in enumerate(train_loader):
        
        # 创建标签
        real_labels = torch.ones(batch_size, 1).to(device)
        fake_labels = torch.zeros(batch_size, 1).to(device)
        
        # 训练判别器
        # 真实图像
        images = images.to(device)
        outputs = discriminator(images)
        d_loss_real = criterion(outputs, real_labels)
        real_score = outputs
        
        # 生成图像
        noise = torch.randn(batch_size, nz).to(device)
        fake_images = generator(noise)
        outputs = discriminator(fake_images.detach())  # 使用detach()避免计算生成器的梯度
        d_loss_fake = criterion(outputs, fake_labels)
        fake_score = outputs
        
        # 反向传播和优化
        d_loss = d_loss_real + d_loss_fake
        d_optimizer.zero_grad()
        g_optimizer.zero_grad()
        d_loss.backward()
        d_optimizer.step()
        
        # 训练生成器
        fake_images = generator(noise)
        outputs = discriminator(fake_images)
        g_loss = criterion(outputs, real_labels)
        
        # 反向传播和优化
        d_optimizer.zero_grad()
        g_optimizer.zero_grad()
        g_loss.backward()
        g_optimizer.step()
        
        if (i + 1) % 300 == 0:
            print(f'Epoch [{epoch + 1}/{num_epochs}], Step [{i + 1}/{len(train_loader)}], '
                  f'd_loss: {d_loss.item():.4f}, g_loss: {g_loss.item():.4f}, '
                  f'D(x): {real_score.mean().item():.2f}, D(G(z)): {fake_score.mean().item():.2f}')
    
    # 保存生成的图像
    fake_images = fake_images.reshape(fake_images.size(0), 1, 28, 28)
    grid = make_grid(fake_images[:25], nrow=5)
    plt.figure(figsize=(5, 5))
    plt.imshow(grid.permute(1, 2, 0).cpu().detach().numpy())
    plt.axis('off')
    plt.show()

Epoch [1/5], Step [300/938], d_loss: 0.2409, g_loss: 2.2740, D(x): 0.97, D(G(z)): 0.19
Epoch [1/5], Step [600/938], d_loss: 0.0086, g_loss: 7.3796, D(x): 0.99, D(G(z)): 0.00
Epoch [1/5], Step [900/938], d_loss: 0.0122, g_loss: 6.8885, D(x): 1.00, D(G(z)): 0.01


ValueError: Using a target size (torch.Size([64, 1])) that is different to the input size (torch.Size([32, 1])) is deprecated. Please ensure they have the same size.

In [ ]:
# 保存模型
torch.save(generator.state_dict(), 'generator.pth')
torch.save(discriminator.state_dict(), 'discriminator.pth')